In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit.primitives import StatevectorSampler
# Ensure your src folder is in the path so we can import your circuit
import sys
sys.path.append('../src')
from circuits import get_quantum_circuits

# Set seed for reproducibility
algorithm_globals.random_seed = 42
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load 
data_path = '../data/mnist_01_quantum.npz'
data = np.load(data_path)
X, y = data['X'], data['y']

# Select a subset
train_size = 1000
X_train, y_train = X[:train_size], y[:train_size].astype(int)

print(f"Ready to train with {train_size} samples.")
print(f"Feature shape: {X_train.shape}")

Ready to train with 1000 samples.
Feature shape: (1000, 4)


In [3]:
# Get our Quantum Architecture
# We use reps=3 for that 92.9% potential
feature_map, ansatz = get_quantum_circuits(num_qubits=4, reps=3)

# Setup the Optimizer and Sampler (Primitive V2)
optimizer = COBYLA(maxiter=100)
sampler = StatevectorSampler()

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    sampler=sampler
)

print(f"VQC initialized with {ansatz.num_parameters} trainable parameters.")


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


VQC initialized with 16 trainable parameters.


In [ ]:
print("Starting Quantum Training... This may take a few minutes.")
vqc.fit(X_train, y_train)
print("Training Complete!")

# Check Accuracy
train_score = vqc.score(X_train, y_train)
print(f"Final Training Accuracy: {train_score * 100:.2f}%")

Starting Quantum Training... This may take a few minutes.


In [ ]:
# Extract the optimized weights
optimized_weights = vqc.weights.tolist()

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save to JSON
with open('../models/vqc_weights_from_ipynb.json', 'w') as f:
    json.dump(optimized_weights, f)

print("✅ Success! Weights are now saved in /models/vqc_weights.json")